# 🚨 SISTEM PERINGATAN DINI - IMPROVED VERSION

## Perbaikan Utama:
1. ✅ Data cleaning yang proper (bisa bedakan 0 asli vs missing)
2. ✅ Deteksi TREN penurunan prestasi (kunci peringatan dini!)
3. ✅ Sistem alert 3 tingkat (Urgent, Warning, Watch)
4. ✅ Prioritas intervensi berdasarkan perubahan, bukan hanya kondisi saat ini
5. ✅ Lebih banyak fitur untuk analisis

In [27]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import json
import warnings
warnings.filterwarnings('ignore')

In [28]:
# ==========================================
# BAGIAN 1: LOAD & CLEANING DATA (IMPROVED)
# ==========================================

FILE_PATH = '/content/data_murid.csv'

df = pd.read_csv(FILE_PATH, header=None)

# Mapping kolom
DATA_START_ROW = 5
COL_NIS = 1
COL_NAMA = 2
COL_NILAI_START = 6
COL_NILAI_END = 40
COL_SAKIT = 40
COL_IZIN = 41
COL_ALPA = 42

# Buat DataFrame bersih
df_clean = pd.DataFrame()

# Ambil identitas
df_clean['nis'] = df.iloc[DATA_START_ROW:, COL_NIS]
df_clean['nama_siswa'] = df.iloc[DATA_START_ROW:, COL_NAMA]

# ==========================================
# PERBAIKAN: CLEANING TANPA fillna(0) DULU
# ==========================================

df_clean['sakit'] = pd.to_numeric(df.iloc[DATA_START_ROW:, COL_SAKIT], errors='coerce')
df_clean['izin'] = pd.to_numeric(df.iloc[DATA_START_ROW:, COL_IZIN], errors='coerce')
df_clean['alpa'] = pd.to_numeric(df.iloc[DATA_START_ROW:, COL_ALPA], errors='coerce')

# Ambil nilai tanpa fillna dulu
nilai_raw = df.iloc[DATA_START_ROW:, COL_NILAI_START:COL_NILAI_END]
nilai_raw = nilai_raw.apply(pd.to_numeric, errors='coerce')

# Hitung rata-rata nilai dan jumlah nilai valid
df_clean['rata_rata_nilai'] = nilai_raw.mean(axis=1, skipna=True)
df_clean['jumlah_nilai_valid'] = nilai_raw.notna().sum(axis=1)

# CLEANING: Hapus data tidak valid
print(f"Total baris SEBELUM cleaning: {len(df_clean)}")

df_clean = df_clean.dropna(subset=['nama_siswa'])
df_clean = df_clean[df_clean['nama_siswa'].astype(str).str.strip() != '']
df_clean = df_clean.dropna(subset=['sakit', 'izin', 'alpa'], how='all')
df_clean = df_clean[df_clean['jumlah_nilai_valid'] > 0]

print(f"Total baris SETELAH cleaning: {len(df_clean)}")

# Isi NaN yang tersisa dengan 0 (data sudah valid)
df_clean['sakit'] = df_clean['sakit'].fillna(0)
df_clean['izin'] = df_clean['izin'].fillna(0)
df_clean['alpa'] = df_clean['alpa'].fillna(0)

df_clean['total_absensi'] = df_clean['sakit'] + df_clean['izin'] + df_clean['alpa']
df_clean.reset_index(drop=True, inplace=True)

print(f"\n✅ Data berhasil dimuat dan dibersihkan!")
print(f"Total siswa dengan data valid: {len(df_clean)}")

Total baris SEBELUM cleaning: 4995
Total baris SETELAH cleaning: 3119

✅ Data berhasil dimuat dan dibersihkan!
Total siswa dengan data valid: 3119


In [29]:
# ==========================================
# BAGIAN 2: FEATURE ENGINEERING (BARU!) 🚀
# ==========================================

# CATATAN: Untuk sistem peringatan dini yang IDEAL, Anda perlu:
# 1. Data per bulan (bulan 1, bulan 2, dst)
# 2. Data historis (semester lalu)
#
# Karena sekarang kita hanya punya data akhir semester,
# kita akan simulasikan dengan asumsi:
# - 50% pertama nilai = periode awal semester
# - 50% terakhir nilai = periode akhir semester

print("\n===== FEATURE ENGINEERING =====")

# Ambil data nilai per kolom untuk analisis tren
nilai_data = df.iloc[DATA_START_ROW:, COL_NILAI_START:COL_NILAI_END].iloc[df_clean.index - DATA_START_ROW]
nilai_numeric = nilai_data.apply(pd.to_numeric, errors='coerce')

# Hitung jumlah kolom nilai
jumlah_kolom_nilai = COL_NILAI_END - COL_NILAI_START
tengah = jumlah_kolom_nilai // 2

# Simulasi: Periode awal vs akhir
nilai_periode_awal = nilai_numeric.iloc[:, :tengah]
nilai_periode_akhir = nilai_numeric.iloc[:, tengah:]

df_clean['rata_nilai_periode_awal'] = nilai_periode_awal.mean(axis=1, skipna=True)
df_clean['rata_nilai_periode_akhir'] = nilai_periode_akhir.mean(axis=1, skipna=True)

# ==========================================
# FITUR BARU 1: TREN NILAI (KUNCI PERINGATAN DINI!) 🎯
# ==========================================
df_clean['perubahan_nilai'] = df_clean['rata_nilai_periode_akhir'] - df_clean['rata_nilai_periode_awal']
df_clean['persentase_perubahan'] = (df_clean['perubahan_nilai'] / df_clean['rata_nilai_periode_awal']) * 100

# Flag penurunan tajam (turun >15%)
df_clean['flag_penurunan_tajam'] = df_clean['persentase_perubahan'] < -15

# ==========================================
# FITUR BARU 2: VOLATILITAS NILAI
# ==========================================
df_clean['volatilitas_nilai'] = nilai_numeric.std(axis=1, skipna=True)

# ==========================================
# FITUR BARU 3: RASIO KEHADIRAN
# ==========================================
# Asumsi: 1 semester = 120 hari efektif
TOTAL_HARI_EFEKTIF = 120
df_clean['hari_hadir'] = TOTAL_HARI_EFEKTIF - df_clean['total_absensi']
df_clean['rasio_kehadiran'] = (df_clean['hari_hadir'] / TOTAL_HARI_EFEKTIF) * 100

# ==========================================
# FITUR BARU 4: KONSISTENSI NILAI
# ==========================================
# Siswa dengan nilai sangat fluktuatif = bermasalah
df_clean['konsistensi'] = np.where(df_clean['volatilitas_nilai'] < 10, 'Konsisten',
                          np.where(df_clean['volatilitas_nilai'] < 20, 'Cukup Stabil', 'Tidak Stabil'))

print("\n✅ Feature Engineering selesai!")
print(f"\nFitur baru yang ditambahkan:")
print("1. Tren nilai (perubahan periode awal vs akhir)")
print("2. Volatilitas nilai (konsistensi)")
print("3. Rasio kehadiran")
print("4. Flag penurunan tajam")

print(f"\n=== Statistik Perubahan Nilai ===")
print(f"Siswa dengan penurunan tajam (>15%): {df_clean['flag_penurunan_tajam'].sum()}")
print(f"Rata-rata perubahan nilai: {df_clean['perubahan_nilai'].mean():.2f}")
print(f"Perubahan terbesar (turun): {df_clean['perubahan_nilai'].min():.2f}")
print(f"Perubahan terbesar (naik): {df_clean['perubahan_nilai'].max():.2f}")


===== FEATURE ENGINEERING =====

✅ Feature Engineering selesai!

Fitur baru yang ditambahkan:
1. Tren nilai (perubahan periode awal vs akhir)
2. Volatilitas nilai (konsistensi)
3. Rasio kehadiran
4. Flag penurunan tajam

=== Statistik Perubahan Nilai ===
Siswa dengan penurunan tajam (>15%): 7
Rata-rata perubahan nilai: 20.71
Perubahan terbesar (turun): -7.61
Perubahan terbesar (naik): 152.75


In [30]:
# ==========================================
# BAGIAN 3: AI CLUSTERING (IMPROVED)
# ==========================================

print("\n===== AI CLUSTERING ====")

# Identifikasi fitur yang akan digunakan untuk clustering
clustering_features = [
    'total_absensi',
    'rata_rata_nilai',
    'perubahan_nilai',
    'volatilitas_nilai',
    'rasio_kehadiran'
]

# PERBAIKAN: Hapus baris dengan NaN dari df_clean untuk fitur clustering
# Ini akan memastikan bahwa X tidak mengandung NaN
initial_len_df_clean = len(df_clean)
df_clean_clustered = df_clean.dropna(subset=clustering_features).copy()
num_dropped_for_clustering = initial_len_df_clean - len(df_clean_clustered)
if num_dropped_for_clustering > 0:
    print(f"Peringatan: {num_dropped_for_clustering} siswa dikeluarkan dari clustering karena data tidak lengkap untuk fitur-fitur clustering.")


X = df_clean_clustered[clustering_features]

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-Means dengan 5 cluster (lebih detail)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df_clean_clustered['cluster_id'] = cluster_labels

# Auto-labeling berdasarkan rata-rata nilai DAN perubahan
cluster_stats = df_clean_clustered.groupby('cluster_id').agg({
    'rata_rata_nilai': 'mean',
    'perubahan_nilai': 'mean',
    'total_absensi': 'mean'
}).round(2)

print("\n=== Statistik per Cluster ===")
print(cluster_stats)

# Labeling berdasarkan nilai dan tren
# Cluster dengan nilai rendah ATAU penurunan besar = Berisiko Tinggi
cluster_stats['risk_score'] = (
    -cluster_stats['rata_rata_nilai'] +  # Nilai rendah = risiko tinggi
    -cluster_stats['perubahan_nilai'] * 2 +  # Penurunan = risiko tinggi (bobot 2x)
    cluster_stats['total_absensi'] * 0.5  # Absensi tinggi = risiko tinggi
)

sorted_clusters = cluster_stats.sort_values('risk_score', ascending=False).index.tolist()

# Mapping dengan 5 kategori
risk_map = {
    sorted_clusters[0]: 'Berisiko Sangat Tinggi',
    sorted_clusters[1]: 'Berisiko Tinggi',
    sorted_clusters[2]: 'Berisiko Sedang',
    sorted_clusters[3]: 'Waspada',
    sorted_clusters[4]: 'Aman'
}

df_clean_clustered['status_risiko_cluster'] = df_clean_clustered['cluster_id'].map(risk_map)

# Gabungkan hasil clustering kembali ke df_clean asli
# Inisialisasi kolom baru di df_clean asli dengan NaN
df_clean['cluster_id'] = np.nan
df_clean['status_risiko_cluster'] = 'Tidak Dikategorikan' # Default for unclustered students

# Isi nilai untuk siswa yang berhasil di-cluster
df_clean.loc[df_clean_clustered.index, 'cluster_id'] = df_clean_clustered['cluster_id']
df_clean.loc[df_clean_clustered.index, 'status_risiko_cluster'] = df_clean_clustered['status_risiko_cluster']


print("\n✅ Clustering selesai dengan 5 kategori risiko!")


===== AI CLUSTERING ====
Peringatan: 26 siswa dikeluarkan dari clustering karena data tidak lengkap untuk fitur-fitur clustering.

=== Statistik per Cluster ===
            rata_rata_nilai  perubahan_nilai  total_absensi
cluster_id                                                 
0                     78.09             1.84           9.40
1                     80.57           142.53           5.49
2                     69.16             8.77          19.52
3                      4.83            27.52          52.00
4                     83.13             1.80           2.81

✅ Clustering selesai dengan 5 kategori risiko!


In [31]:
# ==========================================
# BAGIAN 4: SISTEM PERINGATAN DINI (BARU!) 🚨
# ==========================================

print("\n===== SISTEM PERINGATAN DINI =====")

# LEVEL 1: URGENT - Butuh intervensi SEGERA!
# Kriteria: Penurunan tajam ATAU nilai sangat rendah dengan absensi tinggi
urgent_mask = (
    (df_clean['flag_penurunan_tajam'] == True) |  # Turun >15%
    ((df_clean['rata_rata_nilai'] < 50) & (df_clean['total_absensi'] > 30))
)

# LEVEL 2: WARNING - Perlu monitoring ketat
# Kriteria: Nilai rendah ATAU absensi tinggi (tapi tidak keduanya)
warning_mask = (
    ~urgent_mask &  # Bukan urgent
    (
        (df_clean['rata_rata_nilai'] < 60) |
        (df_clean['total_absensi'] > 20) |
        (df_clean['perubahan_nilai'] < -10)  # Turun >10 poin
    )
)

# LEVEL 3: WATCH - Perlu perhatian
# Kriteria: Nilai agak rendah atau volatilitas tinggi
watch_mask = (
    ~urgent_mask & ~warning_mask &
    (
        (df_clean['rata_rata_nilai'] < 70) |
        (df_clean['volatilitas_nilai'] > 20)
    )
)

# LEVEL 4: NORMAL - Tidak ada masalah
normal_mask = ~(urgent_mask | warning_mask | watch_mask)

# Assign status
df_clean['status_peringatan'] = 'NORMAL'
df_clean.loc[watch_mask, 'status_peringatan'] = 'WATCH'
df_clean.loc[warning_mask, 'status_peringatan'] = 'WARNING'
df_clean.loc[urgent_mask, 'status_peringatan'] = '🚨 URGENT'

# Prioritas intervensi (1 = paling urgent)
priority_map = {
    '🚨 URGENT': 1,
    'WARNING': 2,
    'WATCH': 3,
    'NORMAL': 4
}
df_clean['prioritas_intervensi'] = df_clean['status_peringatan'].map(priority_map)

# Risk score untuk ranking dalam kelompok
df_clean['risk_score'] = (
    -df_clean['rata_rata_nilai'] * 0.3 +  # Nilai rendah = skor tinggi
    -df_clean['perubahan_nilai'] * 0.4 +  # Penurunan = skor tinggi (bobot terbesar!)
    df_clean['total_absensi'] * 0.2 +     # Absensi tinggi = skor tinggi
    df_clean['volatilitas_nilai'] * 0.1   # Tidak stabil = skor tinggi
)

# Normalisasi risk_score ke 0-100
df_clean['risk_score'] = (
    (df_clean['risk_score'] - df_clean['risk_score'].min()) /
    (df_clean['risk_score'].max() - df_clean['risk_score'].min()) * 100
)

# Sort by prioritas dan risk score
df_clean = df_clean.sort_values(['prioritas_intervensi', 'risk_score'], ascending=[True, False])
df_clean.reset_index(drop=True, inplace=True)

# Statistik
print(f"\n=== RINGKASAN PERINGATAN ===")
print(f"🚨 URGENT: {urgent_mask.sum()} siswa (Butuh intervensi SEGERA!)")
print(f"⚠️  WARNING: {warning_mask.sum()} siswa (Perlu monitoring ketat)")
print(f"👁️  WATCH: {watch_mask.sum()} siswa (Perlu perhatian)")
print(f"✅ NORMAL: {normal_mask.sum()} siswa (Tidak ada masalah)")

print(f"\n=== TOP 10 SISWA PALING BERISIKO ===")
top_10 = df_clean.head(10)[[
    'nama_siswa', 'status_peringatan', 'rata_rata_nilai',
    'perubahan_nilai', 'total_absensi', 'risk_score'
]]
print(top_10.to_string(index=False))


===== SISTEM PERINGATAN DINI =====

=== RINGKASAN PERINGATAN ===
🚨 URGENT: 18 siswa (Butuh intervensi SEGERA!)
⚠️  WARNING: 141 siswa (Perlu monitoring ketat)
👁️  WATCH: 582 siswa (Perlu perhatian)
✅ NORMAL: 2378 siswa (Tidak ada masalah)

=== TOP 10 SISWA PALING BERISIKO ===
         nama_siswa status_peringatan  rata_rata_nilai  perubahan_nilai  total_absensi  risk_score
AHMAD MAULANA AFIIF          🚨 URGENT         4.833333        -1.873303           52.0  100.000000
AHMAD MAULANA AFIIF          🚨 URGENT         4.833333        -1.230769           52.0   99.692937
AHMAD MAULANA AFIIF          🚨 URGENT         4.833333         0.820833           52.0   99.429315
AHMAD MAULANA AFIIF          🚨 URGENT         4.833333         1.678733           52.0   98.366096
AHMAD MAULANA AFIIF          🚨 URGENT         4.833333         0.945701           52.0   97.778318
AHMAD MAULANA AFIIF          🚨 URGENT         4.833333         1.615385           52.0   97.370064
AHMAD MAULANA AFIIF          

In [32]:
# ==========================================
# BAGIAN 5: REKOMENDASI INTERVENSI (BARU!) 💡
# ==========================================

print("\n===== REKOMENDASI INTERVENSI ====")

# Check if 'status_peringatan' column exists. If not, the upstream cell (early_warning_system) might not have been executed or completed successfully.
if 'status_peringatan' not in df_clean.columns:
    print("❌ ERROR: Kolom 'status_peringatan' tidak ditemukan di DataFrame df_clean.")
    print("        Pastikan sel 'Sistem Peringatan Dini' (early_warning_system) telah dieksekusi dengan benar.")
else:
    def buat_rekomendasi(row):
        rekomendasi = []

        # Berdasarkan penurunan nilai
        if row['flag_penurunan_tajam']:
            rekomendasi.append("URGENT: Investigasi segera penyebab penurunan drastis")
            rekomendasi.append("Panggil siswa dan orang tua untuk konseling")

        # Berdasarkan nilai
        if row['rata_rata_nilai'] < 50:
            rekomendasi.append("Bimbingan belajar intensif (minimal 2x seminggu)")
            rekomendasi.append("Evaluasi metode belajar siswa")
        elif row['rata_rata_nilai'] < 70:
            rekomendasi.append("Bimbingan belajar reguler (1x seminggu)")

        # Berdasarkan absensi
        if row['total_absensi'] > 30:
            rekomendasi.append("URGENT: Investigasi penyebab sering tidak hadir")
            rekomendasi.append("Home visit oleh wali kelas atau BK")
        elif row['total_absensi'] > 15:
            rekomendasi.append("Konseling terkait kedisiplinan")

        # Berdasarkan volatilitas
        if row['volatilitas_nilai'] > 20:
            rekomendasi.append("Evaluasi kestabilan emosi/mental siswa")
            rekomendasi.append("Konseling BK untuk mengetahui masalah pribadi")

        # Berdasarkan status
        if row['status_peringatan'] == '🚨 URGENT':
            rekomendasi.append("TINDAK LANJUT DALAM 3 HARI!")
        elif row['status_peringatan'] == 'WARNING':
            rekomendasi.append("Monitoring ketat setiap minggu")
        elif row['status_peringatan'] == 'WATCH':
            rekomendasi.append("Monitoring berkala setiap 2 minggu")

        return " | ".join(rekomendasi) if rekomendasi else "Pertahankan performa"

    df_clean['rekomendasi_intervensi'] = df_clean.apply(buat_rekomendasi, axis=1)

    print("\n=== CONTOH REKOMENDASI (5 siswa teratas) ===")
    for idx, row in df_clean.head(5).iterrows():
        print(f"\n{idx+1}. {row['nama_siswa']} ({row['status_peringatan']})")
        print(f"   Risk Score: {row['risk_score']:.1f}/100")
        print(f"   Nilai: {row['rata_rata_nilai']:.1f}, Perubahan: {row['perubahan_nilai']:.1f}, Absensi: {row['total_absensi']:.0f}")
        print(f"   📋 Rekomendasi:")
        for rec in row['rekomendasi_intervensi'].split(" | "):
            print(f"      - {rec}")

    print("\n✅ Rekomendasi intervensi berhasil dibuat!")


===== REKOMENDASI INTERVENSI ====

=== CONTOH REKOMENDASI (5 siswa teratas) ===

1. AHMAD MAULANA AFIIF (🚨 URGENT)
   Risk Score: 100.0/100
   Nilai: 4.8, Perubahan: -1.9, Absensi: 52
   📋 Rekomendasi:
      - Bimbingan belajar intensif (minimal 2x seminggu)
      - Evaluasi metode belajar siswa
      - URGENT: Investigasi penyebab sering tidak hadir
      - Home visit oleh wali kelas atau BK
      - TINDAK LANJUT DALAM 3 HARI!

2. AHMAD MAULANA AFIIF (🚨 URGENT)
   Risk Score: 99.7/100
   Nilai: 4.8, Perubahan: -1.2, Absensi: 52
   📋 Rekomendasi:
      - Bimbingan belajar intensif (minimal 2x seminggu)
      - Evaluasi metode belajar siswa
      - URGENT: Investigasi penyebab sering tidak hadir
      - Home visit oleh wali kelas atau BK
      - TINDAK LANJUT DALAM 3 HARI!

3. AHMAD MAULANA AFIIF (🚨 URGENT)
   Risk Score: 99.4/100
   Nilai: 4.8, Perubahan: 0.8, Absensi: 52
   📋 Rekomendasi:
      - Bimbingan belajar intensif (minimal 2x seminggu)
      - Evaluasi metode belajar siswa
 

In [33]:
# ==========================================
# BAGIAN 6: EXPORT HASIL
# ==========================================

# Export ke Excel (lengkap)
output_excel = 'Hasil_Sistem_Peringatan_Dini_IMPROVED.xlsx'
df_clean.to_excel(output_excel, index=False)
print(f"\n✅ File lengkap berhasil disimpan: {output_excel}")

# Export khusus siswa URGENT (untuk tindak lanjut segera)
df_urgent = df_clean[df_clean['status_peringatan'] == '🚨 URGENT']
if len(df_urgent) > 0:
    urgent_excel = 'SISWA_URGENT_Tindak_Lanjut_Segera.xlsx'
    df_urgent[[
        'nama_siswa', 'nis', 'rata_rata_nilai', 'perubahan_nilai',
        'total_absensi', 'risk_score', 'rekomendasi_intervensi'
    ]].to_excel(urgent_excel, index=False)
    print(f"✅ File siswa URGENT berhasil disimpan: {urgent_excel}")

# Export ke JSON
output_json = 'hasil_peringatan_dini.json'
hasil = {
    'metadata': {
        'total_siswa': len(df_clean),
        'urgent': int(urgent_mask.sum()),
        'warning': int(warning_mask.sum()),
        'watch': int(watch_mask.sum()),
        'normal': int(normal_mask.sum())
    },
    'siswa_urgent': df_clean[df_clean['status_peringatan'] == '🚨 URGENT'][[
        'nis', 'nama_siswa', 'rata_rata_nilai', 'perubahan_nilai',
        'total_absensi', 'risk_score', 'rekomendasi_intervensi'
    ]].to_dict('records'),
    'data_lengkap': df_clean[[
        'nis', 'nama_siswa', 'rata_rata_nilai', 'perubahan_nilai',
        'total_absensi', 'status_peringatan', 'risk_score',
        'status_risiko_cluster', 'rekomendasi_intervensi'
    ]].to_dict('records')
}

with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(hasil, f, ensure_ascii=False, indent=2)

print(f"✅ File JSON berhasil disimpan: {output_json}")

print(f"\n{'='*60}")
print("📊 RINGKASAN AKHIR SISTEM PERINGATAN DINI")
print(f"{'='*60}")
print(f"Total Siswa Dianalisis: {len(df_clean)}")
print(f"\n🚨 URGENT (Tindak lanjut dalam 3 hari): {urgent_mask.sum()} siswa")
print(f"⚠️  WARNING (Monitoring ketat): {warning_mask.sum()} siswa")
print(f"👁️  WATCH (Perlu perhatian): {watch_mask.sum()} siswa")
print(f"✅ NORMAL (Tidak ada masalah): {normal_mask.sum()} siswa")
print(f"\n📈 Siswa dengan penurunan tajam (>15%): {df_clean['flag_penurunan_tajam'].sum()}")
print(f"📉 Rata-rata perubahan nilai: {df_clean['perubahan_nilai'].mean():.2f}")
print(f"{'='*60}")


✅ File lengkap berhasil disimpan: Hasil_Sistem_Peringatan_Dini_IMPROVED.xlsx
✅ File siswa URGENT berhasil disimpan: SISWA_URGENT_Tindak_Lanjut_Segera.xlsx
✅ File JSON berhasil disimpan: hasil_peringatan_dini.json

📊 RINGKASAN AKHIR SISTEM PERINGATAN DINI
Total Siswa Dianalisis: 3119

🚨 URGENT (Tindak lanjut dalam 3 hari): 18 siswa
⚠️  WARNING (Monitoring ketat): 141 siswa
👁️  WATCH (Perlu perhatian): 582 siswa
✅ NORMAL (Tidak ada masalah): 2378 siswa

📈 Siswa dengan penurunan tajam (>15%): 7
📉 Rata-rata perubahan nilai: 20.71


# Task
Split the `df_clean` DataFrame into an 80% training set and a 20% testing set using `train_test_split` from `sklearn.model_selection`, setting a random state for reproducibility, and then print the shapes of the resulting training and testing DataFrames.

## Split Data into Training and Testing Sets

### Subtask:
Split the `df_clean` DataFrame into an 80% training set and a 20% testing set using `train_test_split` from `sklearn.model_selection`.


**Reasoning**:
To split the data into training and testing sets, I need to import the `train_test_split` function from `sklearn.model_selection`.



In [34]:
from sklearn.model_selection import train_test_split

# Split the df_clean DataFrame into training (80%) and testing (20%) sets
df_train, df_test = train_test_split(df_clean, test_size=0.2, random_state=42)

print(f"Ukuran training set: {len(df_train)} siswa")
print(f"Ukuran testing set: {len(df_test)} siswa")
print("\n✅ Data berhasil dibagi menjadi training dan testing set.")

Ukuran training set: 2495 siswa
Ukuran testing set: 624 siswa

✅ Data berhasil dibagi menjadi training dan testing set.


## Display Training and Testing Data Information

### Subtask:
Print the shapes (number of rows and columns) of the training and testing DataFrames to confirm the successful split.


**Reasoning**:
To confirm the successful split, I need to print the shapes (number of rows and columns) of both the training and testing DataFrames.



In [35]:
print(f"Shape of training set (rows, columns): {df_train.shape}")
print(f"Shape of testing set (rows, columns): {df_test.shape}")

Shape of training set (rows, columns): (2495, 23)
Shape of testing set (rows, columns): (624, 23)


## Final Task

### Subtask:
Confirm that the data has been successfully split into training and testing sets and ask the user if they want to proceed with building a predictive model or another analysis.


## Summary:

### Q&A
Yes, the data has been successfully split into training and testing sets.

### Data Analysis Key Findings
*   The `df_clean` DataFrame, initially containing 3119 entries, was successfully split into an 80% training set and a 20% testing set using `train_test_split` with a `random_state` of 42.
*   The training set (`df_train`) consists of 2495 rows and 23 columns.
*   The testing set (`df_test`) consists of 624 rows and 23 columns.

### Insights or Next Steps
*   The data is now prepared for subsequent machine learning tasks, such as building and evaluating a predictive model.
*   Confirm with the user whether to proceed with building a predictive model or conduct another analysis using the split datasets.


## Refactor Warning System and Recommendations

### Subtask:
Create a Python function, `apply_early_warning_system_and_recommendations`, that encapsulates the logic from the `early_warning_system` and `recommendations` cells. This function will take a DataFrame as input, apply the warning system logic, calculate risk scores, generate recommendations, and return the processed DataFrame. This will make the code reusable for both training and testing sets.


**Reasoning**:
The subtask requires encapsulating the early warning system and recommendation logic into a reusable Python function. I will create the `apply_early_warning_system_and_recommendations` function, which takes a DataFrame, applies all the criteria for alerts, calculates risk scores, generates specific recommendations, and returns the modified DataFrame.



In [22]:
import numpy as np

def apply_early_warning_system_and_recommendations(df_input):
    df = df_input.copy()

    # ==========================================
    # SISTEM PERINGATAN DINI
    # ==========================================

    # LEVEL 1: URGENT - Butuh intervensi SEGERA!
    # Kriteria: Penurunan tajam ATAU nilai sangat rendah dengan absensi tinggi
    urgent_mask = (
        (df['flag_penurunan_tajam'] == True) |  # Turun >15%
        ((df['rata_rata_nilai'] < 50) & (df['total_absensi'] > 30))
    )

    # LEVEL 2: WARNING - Perlu monitoring ketat
    # Kriteria: Nilai rendah ATAU absensi tinggi (tapi tidak keduanya)
    warning_mask = (
        ~urgent_mask &  # Bukan urgent
        (
            (df['rata_rata_nilai'] < 60) |
            (df['total_absensi'] > 20) |
            (df['perubahan_nilai'] < -10)  # Turun >10 poin
        )
    )

    # LEVEL 3: WATCH - Perlu perhatian
    # Kriteria: Nilai agak rendah atau volatilitas tinggi
    watch_mask = (
        ~urgent_mask & ~warning_mask &
        (
            (df['rata_rata_nilai'] < 70) |
            (df['volatilitas_nilai'] > 20)
        )
    )

    # LEVEL 4: NORMAL - Tidak ada masalah
    normal_mask = ~(urgent_mask | warning_mask | watch_mask)

    # Assign status
    df['status_peringatan'] = 'NORMAL'
    df.loc[watch_mask, 'status_peringatan'] = 'WATCH'
    df.loc[warning_mask, 'status_peringatan'] = 'WARNING'
    df.loc[urgent_mask, 'status_peringatan'] = '🚨 URGENT'

    # Prioritas intervensi (1 = paling urgent)
    priority_map = {
        '🚨 URGENT': 1,
        'WARNING': 2,
        'WATCH': 3,
        'NORMAL': 4
    }
    df['prioritas_intervensi'] = df['status_peringatan'].map(priority_map)

    # Risk score untuk ranking dalam kelompok
    df['risk_score'] = (
        -df['rata_rata_nilai'] * 0.3 +  # Nilai rendah = skor tinggi
        -df['perubahan_nilai'] * 0.4 +  # Penurunan = skor tinggi (bobot terbesar!)
        df['total_absensi'] * 0.2 +     # Absensi tinggi = skor tinggi
        df['volatilitas_nilai'] * 0.1   # Tidak stabil = skor tinggi
    )

    # Normalisasi risk_score ke 0-100
    min_risk_score = df['risk_score'].min()
    max_risk_score = df['risk_score'].max()

    if max_risk_score == min_risk_score:
        df['risk_score'] = 0 # All scores are the same, normalize to 0
    else:
        df['risk_score'] = (
            (df['risk_score'] - min_risk_score) /
            (max_risk_score - min_risk_score) * 100
        )

    # Sort by prioritas dan risk score
    df = df.sort_values(['prioritas_intervensi', 'risk_score'], ascending=[True, False])
    df.reset_index(drop=True, inplace=True)

    # ==========================================
    # REKOMENDASI INTERVENSI
    # ==========================================

    def buat_rekomendasi(row):
        rekomendasi = []

        # Berdasarkan penurunan nilai
        if row['flag_penurunan_tajam']:
            rekomendasi.append("URGENT: Investigasi segera penyebab penurunan drastis")
            rekomendasi.append("Panggil siswa dan orang tua untuk konseling")

        # Berdasarkan nilai
        if row['rata_rata_nilai'] < 50:
            rekomendasi.append("Bimbingan belajar intensif (minimal 2x seminggu)")
            rekomendasi.append("Evaluasi metode belajar siswa")
        elif row['rata_rata_nilai'] < 70:
            rekomendasi.append("Bimbingan belajar reguler (1x seminggu)")

        # Berdasarkan absensi
        if row['total_absensi'] > 30:
            rekomendasi.append("URGENT: Investigasi penyebab sering tidak hadir")
            rekomendasi.append("Home visit oleh wali kelas atau BK")
        elif row['total_absensi'] > 15:
            rekomendasi.append("Konseling terkait kedisiplinan")

        # Berdasarkan volatilitas
        if row['volatilitas_nilai'] > 20:
            rekomendasi.append("Evaluasi kestabilan emosi/mental siswa")
            rekomendasi.append("Konseling BK untuk mengetahui masalah pribadi")

        # Berdasarkan status
        if row['status_peringatan'] == '🚨 URGENT':
            rekomendasi.append("TINDAK LANJUT DALAM 3 HARI!")
        elif row['status_peringatan'] == 'WARNING':
            rekomendasi.append("Monitoring ketat setiap minggu")
        elif row['status_peringatan'] == 'WATCH':
            rekomendasi.append("Monitoring berkala setiap 2 minggu")

        return " | ".join(rekomendasi) if rekomendasi else "Pertahankan performa"

    df['rekomendasi_intervensi'] = df.apply(buat_rekomendasi, axis=1)

    return df

print("Function `apply_early_warning_system_and_recommendations` defined successfully.")

Function `apply_early_warning_system_and_recommendations` defined successfully.


**Reasoning**:
Now that the `apply_early_warning_system_and_recommendations` function is defined, I will apply it to the `df_train` and `df_test` DataFrames as instructed, storing the results in `df_train_processed` and `df_test_processed`.



In [23]:
print("\n===== Menerapkan Sistem Peringatan Dini & Rekomendasi ke Data Training =====")
df_train_processed = apply_early_warning_system_and_recommendations(df_train)
print(f"✅ Sistem peringatan dini diterapkan pada {len(df_train_processed)} siswa di training set.")

print("\n===== Menerapkan Sistem Peringatan Dini & Rekomendasi ke Data Testing =====")
df_test_processed = apply_early_warning_system_and_recommendations(df_test)
print(f"✅ Sistem peringatan dini diterapkan pada {len(df_test_processed)} siswa di testing set.")


===== Menerapkan Sistem Peringatan Dini & Rekomendasi ke Data Training =====
✅ Sistem peringatan dini diterapkan pada 2495 siswa di training set.

===== Menerapkan Sistem Peringatan Dini & Rekomendasi ke Data Testing =====
✅ Sistem peringatan dini diterapkan pada 624 siswa di testing set.


**Reasoning**:
To summarize the results for the training set, I will print the distribution of warning statuses and then display the details of the top 5 riskiest students from `df_train_processed`, including their risk scores and intervention recommendations.



In [24]:
print("\n===== RINGKASAN HASIL UNTUK TRAINING SET ====")
summary_train = df_train_processed['status_peringatan'].value_counts()
print(f"\n--- Distribusi Peringatan Training Set ---")
print(summary_train)

print(f"\n--- TOP 5 SISWA PALING BERISIKO (Training Set) ---")
top_5_train = df_train_processed.head(5)
for idx, row in top_5_train.iterrows():
    print(f"\n{idx+1}. {row['nama_siswa']} ({row['status_peringatan']})")
    print(f"   Risk Score: {row['risk_score']:.1f}/100")
    print(f"   Nilai: {row['rata_rata_nilai']:.1f}, Perubahan: {row['perubahan_nilai']:.1f}, Absensi: {row['total_absensi']:.0f}")
    print(f"   📋 Rekomendasi:")
    for rec in row['rekomendasi_intervensi'].split(" | "):
        print(f"      - {rec}")


===== RINGKASAN HASIL UNTUK TRAINING SET ====

--- Distribusi Peringatan Training Set ---
status_peringatan
NORMAL      1924
WATCH        449
WARNING      107
🚨 URGENT      15
Name: count, dtype: int64

--- TOP 5 SISWA PALING BERISIKO (Training Set) ---

1. AHMAD MAULANA AFIIF (🚨 URGENT)
   Risk Score: 100.0/100
   Nilai: 4.8, Perubahan: -1.2, Absensi: 52
   📋 Rekomendasi:
      - Bimbingan belajar intensif (minimal 2x seminggu)
      - Evaluasi metode belajar siswa
      - URGENT: Investigasi penyebab sering tidak hadir
      - Home visit oleh wali kelas atau BK
      - TINDAK LANJUT DALAM 3 HARI!

2. AHMAD MAULANA AFIIF (🚨 URGENT)
   Risk Score: 99.7/100
   Nilai: 4.8, Perubahan: 0.8, Absensi: 52
   📋 Rekomendasi:
      - Bimbingan belajar intensif (minimal 2x seminggu)
      - Evaluasi metode belajar siswa
      - URGENT: Investigasi penyebab sering tidak hadir
      - Home visit oleh wali kelas atau BK
      - TINDAK LANJUT DALAM 3 HARI!

3. AHMAD MAULANA AFIIF (🚨 URGENT)
   Risk 

**Reasoning**:
To summarize the results for the testing set, I will print the distribution of warning statuses and then display the details of the top 5 riskiest students from `df_test_processed`, including their risk scores and intervention recommendations.



In [25]:
print("\n===== RINGKASAN HASIL UNTUK TESTING SET ====")
summary_test = df_test_processed['status_peringatan'].value_counts()
print(f"\n--- Distribusi Peringatan Testing Set ---")
print(summary_test)

print(f"\n--- TOP 5 SISWA PALING BERISIKO (Testing Set) ---")
top_5_test = df_test_processed.head(5)
for idx, row in top_5_test.iterrows():
    print(f"\n{idx+1}. {row['nama_siswa']} ({row['status_peringatan']})")
    print(f"   Risk Score: {row['risk_score']:.1f}/100")
    print(f"   Nilai: {row['rata_rata_nilai']:.1f}, Perubahan: {row['perubahan_nilai']:.1f}, Absensi: {row['total_absensi']:.0f}")
    print(f"   │ Rekomendasi:")
    for rec in row['rekomendasi_intervensi'].split(" | "):
        print(f"      - {rec}")


===== RINGKASAN HASIL UNTUK TESTING SET ====

--- Distribusi Peringatan Testing Set ---
status_peringatan
NORMAL      454
WATCH       133
WARNING      34
🚨 URGENT      3
Name: count, dtype: int64

--- TOP 5 SISWA PALING BERISIKO (Testing Set) ---

1. AHMAD MAULANA AFIIF (🚨 URGENT)
   Risk Score: 100.0/100
   Nilai: 4.8, Perubahan: -1.9, Absensi: 52
   │ Rekomendasi:
      - Bimbingan belajar intensif (minimal 2x seminggu)
      - Evaluasi metode belajar siswa
      - URGENT: Investigasi penyebab sering tidak hadir
      - Home visit oleh wali kelas atau BK
      - TINDAK LANJUT DALAM 3 HARI!

2. ALDIANSYAH AKBAR (🚨 URGENT)
   Risk Score: 46.7/100
   Nilai: 79.9, Perubahan: -7.2, Absensi: 3
   │ Rekomendasi:
      - URGENT: Investigasi segera penyebab penurunan drastis
      - Panggil siswa dan orang tua untuk konseling
      - TINDAK LANJUT DALAM 3 HARI!

3. GEATRY GANIDA (🚨 URGENT)
   Risk Score: 42.7/100
   Nilai: 86.5, Perubahan: -7.2, Absensi: 2
   │ Rekomendasi:
      - URGENT: I